# 03 - RQ2: SHAP Feature Importance

**Notebook version:** v5 -- 2026-07-29

- Apply TreeExplainer SHAP to the best RQ1 model
- Rank features by mean |SHAP value|
- Cross-check against permutation importance
- Report ALE curves for the top features as a complementary, engineer-facing deliverable

**Why SHAP over TreeInterpreter:** TreeInterpreter's Saabas-style decomposition is
provably inconsistent (Lundberg, Erion, & Lee, 2018) -- it can lower a feature's
assigned importance even when that feature's true impact increases. TreeExplainer
was built to fix exactly this, and also (unlike TreeInterpreter) covers both
Random Forest and XGBoost with the same exact algorithm. See `docs/synopsis.docx`,
"Solution to RQ2" for the full rationale, including the parallel discussion of
why LIME was set aside (`src/feature_selection.py` docstring covers PCA).

In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# Also pulls latest changes and prints the commit hash, so you can confirm at a
# glance (against GitHub's commit history) that you're looking at the current version.
import os
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
    else:
        os.chdir(f"/content/{REPO_NAME}")
        !git pull
        os.chdir("/content")
    os.chdir(f"/content/{REPO_NAME}/notebooks")
    !pip install -q -r ../requirements.txt

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import numpy as np
import pandas as pd
import shap

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from explainability import compute_ale, ale_report_for_top_features
from artifacts import load_model, load_json

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))

# Load RQ1's actual best TREE ensemble (not a freshly retrained model) --
# see 02_modeling_rq1.ipynb's "Save results for downstream notebooks" cell.
# Scoped to the best TREE model specifically (not necessarily RQ1's overall
# winner) because TreeExplainer only supports tree-based models -- this
# matches the synopsis's stated methodology, not an arbitrary substitution.
rq1_summary = load_json("rq1_summary")
best_tree_name = rq1_summary["best_tree_model_name"]
best_overall_name = rq1_summary["best_overall_model_name"]
model = load_model("rq1_best_tree")

print(f"Loaded RQ1's best tree ensemble: {best_tree_name}")
if best_overall_name != best_tree_name:
    print(
        f"Note: RQ1's overall best model was {best_overall_name}, not a tree "
        f"ensemble -- this SHAP analysis explains {best_tree_name} specifically, "
        f"per the synopsis's TreeExplainer-based methodology."
    )


## Step 1: SHAP global feature ranking

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

mean_abs_shap = pd.Series(np.abs(sv).mean(axis=0), index=X.columns).sort_values(ascending=False)
top_features = mean_abs_shap.index[:10].tolist()
print("Top 10 features by mean |SHAP value|:")
print(mean_abs_shap.head(10))

## Step 2: Permutation importance cross-check

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X, y, n_repeats=10, random_state=42, scoring='roc_auc')
perm_importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 10 features by permutation importance:")
print(perm_importance.head(10))

overlap = set(top_features) & set(perm_importance.index[:10])
print(f"\nOverlap with SHAP top-10: {len(overlap)}/10")

## Step 3: ALE curves for the top features (complementary deliverable)

SHAP tells you *which* sensors matter; ALE tells you *which direction is bad* --
e.g. "fail probability rises sharply once this sensor exceeds X". Used instead of
Partial Dependence Plots because ALE stays unbiased under correlated features
(Apley & Zhu, 2020), unlike PDP, which assumes feature independence.

In [ ]:
ale_curves = ale_report_for_top_features(model, X, top_features, n_bins=20)

for feat, curve in list(ale_curves.items())[:3]:
    print(f"\nALE curve for {feat}:")
    print(curve)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (feat, curve) in zip(axes.flat, ale_curves.items()):
    ax.plot(curve['bin_edge'], curve['accumulated_local_effect'])
    ax.set_title(feat, fontsize=9)
    ax.axhline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.savefig('../docs/ale_top10_features.png', dpi=150)
plt.show()

## Next steps

- Feed `top_features` (or the nested-CV stabilized version from RQ3) into
  `04_feature_reduction_rq3.ipynb`
- Report the SHAP ranking, permutation-importance overlap, and ALE curves
  together in the capstone write-up as the RQ2 deliverable

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "03_shap_analysis_rq2"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed.
    from google.colab import _message
    ipynb_content = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
    with open(export_path, 'w') as f:
        json.dump(ipynb_content, f)

html_output = f"{NOTEBOOK_NAME}.html"
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print(f"Exported to {html_output}")

if IN_COLAB and result.returncode == 0:
    from google.colab import files
    files.download(html_output)
